# 📊 Notebook 5 — Result Analysis
**Project:** Real-Time Customer Churn Analysis Agent  
**Input:** `data/synthetic_streams/scored_all.csv` · `data/synthetic_streams/alert_log.csv` · `data/synthetic_streams/stream_summary.json` · `models/model_metadata.json` · `data/telco_churn_features.csv`

### What this notebook does
Consolidates everything from all four notebooks into a single comprehensive analysis:

| Section | Analysis |
|---------|----------|
| 1 | Model performance recap |
| 2 | Stream simulation recap |
| 3 | Alert quality deep-dive |
| 4 | Customer risk segmentation |
| 5 | Business impact estimation |
| 6 | Feature importance story |
| 7 | Agent performance profile |
| 8 | Final summary report |

---

## 1. Imports & Load All Results

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
import seaborn as sns
import json, pickle, warnings

from sklearn.metrics import (
    roc_auc_score, confusion_matrix, classification_report,
    precision_score, recall_score, f1_score, accuracy_score
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.dpi'] = 120

TIER_COLORS = {'CRITICAL': '#c0392b', 'HIGH': '#e67e22', 'MEDIUM': '#f1c40f', 'LOW': '#27ae60'}

# ── load everything ────────────────────────────────────────────────────────────
scored_df   = pd.read_csv('../data/synthetic_streams/scored_all.csv',
                          parse_dates=['timestamp'])
alert_df    = pd.read_csv('../data/synthetic_streams/alert_log.csv',
                          parse_dates=['timestamp'])
df_features = pd.read_csv('../data/telco_churn_features.csv')

with open('../data/synthetic_streams/stream_summary.json') as f:
    stream_summary = json.load(f)

with open('../models/model_metadata.json') as f:
    model_meta = json.load(f)

with open('../models/best_model.pkl', 'rb') as f:
    model = pickle.load(f)

THRESHOLD = model_meta['best_threshold']

print('All artefacts loaded ✔')
print(f'  Scored events   : {len(scored_df):,}')
print(f'  Alerts fired    : {len(alert_df):,}')
print(f'  Decision threshold: {THRESHOLD}')

---
## 2. Model Performance Recap

In [ ]:
metrics = model_meta['test_metrics']
metric_df = pd.DataFrame({
    'Metric': list(metrics.keys()),
    'Score' : list(metrics.values())
})

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ── bar chart ─────────────────────────────────────────────────────────────────
bar_colors = ['#e74c3c' if v >= 0.80 else 'steelblue' for v in metric_df['Score']]
axes[0].bar(metric_df['Metric'], metric_df['Score'],
            color=bar_colors, edgecolor='black', width=0.5)
axes[0].axhline(0.80, color='grey', linestyle='--', linewidth=0.9, label='0.80 reference')
axes[0].set_ylim(0, 1.1)
axes[0].set_title('Final Model — Test Set Metrics', fontweight='bold')
axes[0].set_ylabel('Score')
for bar in axes[0].patches:
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.01,
                 f'{bar.get_height():.3f}', ha='center', fontsize=9)
axes[0].legend()

# ── radar / spider chart ──────────────────────────────────────────────────────
radar_metrics = ['accuracy', 'precision', 'recall', 'f1', 'roc_auc']
values = [metrics[m] for m in radar_metrics]
values += values[:1]          # close the polygon
angles = np.linspace(0, 2 * np.pi, len(radar_metrics), endpoint=False).tolist()
angles += angles[:1]

ax_radar = fig.add_subplot(122, polar=True)
ax_radar.plot(angles, values, 'o-', color='#e74c3c', linewidth=2)
ax_radar.fill(angles, values, alpha=0.20, color='#e74c3c')
ax_radar.set_thetagrids(np.degrees(angles[:-1]), radar_metrics)
ax_radar.set_ylim(0, 1)
ax_radar.set_title('Performance Radar', fontweight='bold', pad=15)
ax_radar.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax_radar.set_yticklabels(['0.2','0.4','0.6','0.8','1.0'], fontsize=7)

plt.suptitle(f'Tuned Random Forest — {model_meta["model_type"]}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

---
## 3. Stream Simulation Recap

In [ ]:
# ── KPI cards ─────────────────────────────────────────────────────────────────
kpis = [
    ('Total Events',     stream_summary['total_events'],              '#3498db'),
    ('Alerts Fired',     stream_summary['alerts_fired'],              '#e74c3c'),
    ('Alert Rate',       f"{stream_summary['alert_rate_pct']} %",     '#e67e22'),
    ('Critical',         stream_summary['critical_count'],            '#c0392b'),
    ('Avg Latency',      f"{stream_summary['avg_latency_ms']} ms",    '#27ae60'),
    ('Avg P(Churn)',     stream_summary['avg_churn_probability'],     '#9b59b6'),
]

fig, axes = plt.subplots(1, 6, figsize=(16, 2.2))
for ax, (label, value, color) in zip(axes, kpis):
    ax.set_facecolor(color)
    ax.text(0.5, 0.60, str(value), ha='center', va='center',
            fontsize=20, fontweight='bold', color='white',
            transform=ax.transAxes)
    ax.text(0.5, 0.20, label, ha='center', va='center',
            fontsize=9, color='white', transform=ax.transAxes)
    ax.axis('off')

plt.suptitle('Stream Simulation KPIs', fontsize=12, fontweight='bold', y=1.08)
plt.tight_layout()
plt.show()

In [ ]:
# ── tier breakdown pie ─────────────────────────────────────────────────────────
tier_order  = ['CRITICAL', 'HIGH', 'MEDIUM', 'LOW']
tier_counts = scored_df['risk_tier'].value_counts().reindex(tier_order, fill_value=0)

fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))

wedges, texts, autotexts = axes[0].pie(
    tier_counts.values,
    labels=tier_counts.index,
    colors=[TIER_COLORS[t] for t in tier_counts.index],
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops=dict(edgecolor='white', linewidth=2)
)
axes[0].set_title('Risk Tier Distribution (all events)', fontweight='bold')

# stacked bar over time (hourly)
scored_df['hour'] = scored_df['timestamp'].dt.hour
hourly_tier = scored_df.groupby(['hour', 'risk_tier']).size().unstack(fill_value=0)
hourly_tier = hourly_tier.reindex(columns=tier_order, fill_value=0)
hourly_tier.plot(kind='bar', stacked=True, ax=axes[1],
                 color=[TIER_COLORS[t] for t in tier_order],
                 edgecolor='white', width=0.8)
axes[1].set_title('Risk Tier by Hour of Day', fontweight='bold')
axes[1].set_xlabel('Hour')
axes[1].set_ylabel('Event count')
axes[1].tick_params(axis='x', rotation=0)
axes[1].legend(title='Tier', loc='upper right', fontsize=8)

plt.tight_layout()
plt.show()

---
## 4. Alert Quality Deep-Dive

In [ ]:
# ── inter-alert gap analysis ───────────────────────────────────────────────────
alert_sorted = alert_df.sort_values('timestamp').copy()
alert_sorted['gap_minutes'] = alert_sorted['timestamp'].diff().dt.total_seconds() / 60

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# probability distribution of alerted customers
axes[0].hist(alert_df['churn_probability'], bins=25,
             color='#e74c3c', edgecolor='white', alpha=0.85)
axes[0].axvline(alert_df['churn_probability'].mean(), color='black',
                linestyle='--', label=f'Mean = {alert_df["churn_probability"].mean():.3f}')
axes[0].set_title('Churn Probability — Alerted Customers', fontweight='bold')
axes[0].set_xlabel('P(Churn)')
axes[0].set_ylabel('Count')
axes[0].legend()

# inter-alert gap
gaps = alert_sorted['gap_minutes'].dropna()
axes[1].hist(gaps, bins=20, color='steelblue', edgecolor='white', alpha=0.85)
axes[1].axvline(gaps.mean(), color='red', linestyle='--',
                label=f'Avg gap = {gaps.mean():.1f} min')
axes[1].set_title('Time Gap Between Consecutive Alerts', fontweight='bold')
axes[1].set_xlabel('Minutes')
axes[1].set_ylabel('Count')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# ── Alert tier transition timeline (event-level) ──────────────────────────────
scored_df_sorted = scored_df.sort_values('timestamp')

plt.figure(figsize=(14, 3.5))
tier_y = {'LOW': 1, 'MEDIUM': 2, 'HIGH': 3, 'CRITICAL': 4}
for tier, yval in tier_y.items():
    subset = scored_df_sorted[scored_df_sorted['risk_tier'] == tier]
    plt.scatter(subset['timestamp'], [yval] * len(subset),
                c=TIER_COLORS[tier], s=10, alpha=0.5, label=tier)

plt.yticks([1, 2, 3, 4], ['LOW', 'MEDIUM', 'HIGH', 'CRITICAL'])
plt.xlabel('Time')
plt.title('Risk Tier Timeline — Every Event', fontweight='bold')
plt.legend(loc='upper right', fontsize=8, markerscale=2)
plt.tight_layout()
plt.show()

---
## 5. Customer Risk Segmentation

In [ ]:
# ── segment the FULL dataset by predicted risk ────────────────────────────────
with open('../models/scaler.pkl', 'rb') as f:
    scaler = pickle.load(f)

with open('../data/feature_metadata.json') as f:
    feat_meta = json.load(f)

FEATURE_COLS = feat_meta['feature_columns']
SCALE_COLS   = feat_meta['scaled_columns']

X_full = df_features[FEATURE_COLS].copy()
X_full[SCALE_COLS] = scaler.transform(X_full[SCALE_COLS])

df_features['churn_prob'] = model.predict_proba(X_full)[:, 1]
df_features['risk_tier']  = pd.cut(
    df_features['churn_prob'],
    bins=[0, 0.30, THRESHOLD, 0.75, 1.01],
    labels=['LOW', 'MEDIUM', 'HIGH', 'CRITICAL'],
    include_lowest=True
)

seg_summary = df_features.groupby('risk_tier').agg(
    customers        = ('churn_prob', 'count'),
    avg_churn_prob   = ('churn_prob', 'mean'),
    actual_churn_rate= ('Churn',      'mean')
).round(3)

print('Customer Risk Segments — full dataset:')
display(seg_summary.style.background_gradient(cmap='RdYlGn_r', subset=['avg_churn_prob', 'actual_churn_rate']))

In [ ]:
# ── segment profile plots ─────────────────────────────────────────────────────
profile_cols = ['tenure', 'MonthlyCharges', 'num_services',
                'is_month_to_month', 'has_support', 'risk_score']

# un-scale tenure & MonthlyCharges for readability
df_raw_ref = pd.read_csv('../data/telco_churn_clean.csv')
df_profile = df_features.copy()
df_profile['tenure_raw']         = df_raw_ref['tenure']
df_profile['MonthlyCharges_raw'] = df_raw_ref['MonthlyCharges']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))

plot_pairs = [
    ('tenure_raw',         'Avg Tenure (months)'),
    ('MonthlyCharges_raw', 'Avg Monthly Charges ($)'),
    ('num_services',       'Avg # Services'),
    ('is_month_to_month',  '% Month-to-Month'),
    ('has_support',        '% Has Support'),
    ('risk_score',         'Avg Risk Score (scaled)'),
]

for ax, (col, label) in zip(axes.flatten(), plot_pairs):
    seg_means = df_profile.groupby('risk_tier')[col].mean().reindex(
        ['LOW', 'MEDIUM', 'HIGH', 'CRITICAL'])
    bars = ax.bar(seg_means.index, seg_means.values,
                  color=[TIER_COLORS[t] for t in seg_means.index],
                  edgecolor='black', width=0.5)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() * 1.02,
                f'{bar.get_height():.2f}', ha='center', fontsize=8)
    ax.set_title(label, fontweight='bold')
    ax.set_xlabel('Risk Tier')

plt.suptitle('Customer Segment Profile by Risk Tier', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ── scatter: tenure vs MonthlyCharges coloured by risk tier ───────────────────
plt.figure(figsize=(9, 5))
for tier in ['LOW', 'MEDIUM', 'HIGH', 'CRITICAL']:
    subset = df_profile[df_profile['risk_tier'] == tier]
    plt.scatter(subset['tenure_raw'], subset['MonthlyCharges_raw'],
                c=TIER_COLORS[tier], s=8, alpha=0.4, label=tier)

plt.xlabel('Tenure (months)')
plt.ylabel('Monthly Charges ($)')
plt.title('Customer Risk Map — Tenure vs Monthly Charges', fontweight='bold')
handles = [mpatches.Patch(color=TIER_COLORS[t], label=t) for t in ['LOW','MEDIUM','HIGH','CRITICAL']]
plt.legend(handles=handles, title='Risk Tier', fontsize=8)
plt.tight_layout()
plt.show()

---
## 6. Business Impact Estimation

> **Simple model:**  
> If we intervene on every alerted customer with a retention offer (cost = \$20/customer),  
> and each saved customer has CLV = \$500, what is the net benefit?

We use the model's precision/recall to estimate true positive rate.

In [ ]:
# ── parameters (adjust to your business context) ──────────────────────────────
RETENTION_OFFER_COST = 20      # $ per customer contacted
CLV                  = 500     # $ average customer lifetime value
RETENTION_SUCCESS    = 0.30    # 30% of alerted churners successfully retained

precision = model_meta['test_metrics']['precision']
recall    = model_meta['test_metrics']['recall']

# from the full dataset
total_customers  = len(df_features)
actual_churners  = int(df_features['Churn'].sum())
predicted_alerts = int(df_features['churn_prob'].ge(THRESHOLD).sum())
true_positives   = int(predicted_alerts * precision)
false_positives  = predicted_alerts - true_positives

cost_of_intervention  = predicted_alerts * RETENTION_OFFER_COST
churners_saved        = int(true_positives * RETENTION_SUCCESS)
revenue_saved         = churners_saved * CLV
net_benefit           = revenue_saved - cost_of_intervention
roi_pct               = (net_benefit / cost_of_intervention * 100) if cost_of_intervention > 0 else 0

# ── print ──────────────────────────────────────────────────────────────────────
print('═'*55)
print('  BUSINESS IMPACT ESTIMATION')
print('═'*55)
print(f'  Total customers          : {total_customers:>7,}')
print(f'  Actual churners          : {actual_churners:>7,}')
print(f'  Alerts fired by model    : {predicted_alerts:>7,}')
print(f'  True positives (est.)    : {true_positives:>7,}')
print(f'  False positives (est.)   : {false_positives:>7,}')
print(f'  Churners saved (30% ret.): {churners_saved:>7,}')
print('─'*55)
print(f'  Cost of intervention     : ${cost_of_intervention:>8,.0f}')
print(f'  Revenue saved            : ${revenue_saved:>8,.0f}')
print(f'  Net benefit              : ${net_benefit:>8,.0f}')
print(f'  ROI                      : {roi_pct:>7.1f} %')
print('═'*55)

In [ ]:
# ── sensitivity analysis: ROI vs retention success rate ───────────────────────
retention_rates = np.arange(0.05, 0.65, 0.05)
rois = []
for r in retention_rates:
    rev  = int(true_positives * r) * CLV
    cost = predicted_alerts * RETENTION_OFFER_COST
    rois.append((rev - cost) / cost * 100)

plt.figure(figsize=(8, 4))
plt.plot(retention_rates * 100, rois, 'o-', color='#3498db', linewidth=2.5)
plt.axhline(0, color='grey', linestyle='--', linewidth=0.8)
plt.axvline(RETENTION_SUCCESS * 100, color='red', linestyle='--',
            label=f'Assumed rate ({RETENTION_SUCCESS*100:.0f}%)')
plt.fill_between(retention_rates * 100, rois,
                 where=[r > 0 for r in rois],
                 alpha=0.15, color='green', label='Profitable zone')
plt.xlabel('Retention Success Rate (%)')
plt.ylabel('ROI (%)')
plt.title('ROI Sensitivity vs Retention Success Rate', fontweight='bold')
plt.legend(fontsize=9)
plt.tight_layout()
plt.show()

---
## 7. Feature Importance Story

In [ ]:
imp_df = pd.DataFrame({
    'feature'   : FEATURE_COLS,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

top15 = imp_df.head(15)

engineered_feats = {'num_services', 'avg_monthly_spend', 'charges_per_month_ratio',
                    'risk_score', 'tenure_group', 'has_streaming', 'has_support',
                    'is_month_to_month', 'auto_payment'}

fig, axes = plt.subplots(1, 2, figsize=(15, 6))

# ── horizontal bar ────────────────────────────────────────────────────────────
colors = ['#e67e22' if f in engineered_feats else '#3498db' for f in top15['feature']]
axes[0].barh(top15['feature'][::-1], top15['importance'][::-1],
             color=colors[::-1], edgecolor='grey')
axes[0].set_xlabel('Mean Decrease Impurity')
axes[0].set_title('Top 15 Feature Importances', fontweight='bold')
legend_elements = [
    mpatches.Patch(facecolor='#3498db', label='Original feature'),
    mpatches.Patch(facecolor='#e67e22', label='Engineered feature')
]
axes[0].legend(handles=legend_elements, loc='lower right', fontsize=8)

# ── cumulative importance ─────────────────────────────────────────────────────
cum_imp = imp_df['importance'].cumsum().values
axes[1].plot(range(1, len(cum_imp)+1), cum_imp, color='#e74c3c', linewidth=2)
axes[1].axhline(0.80, color='grey', linestyle='--', label='80% importance')
axes[1].axhline(0.95, color='black', linestyle=':', label='95% importance')
n_80 = np.searchsorted(cum_imp, 0.80) + 1
n_95 = np.searchsorted(cum_imp, 0.95) + 1
axes[1].axvline(n_80, color='grey', linestyle='--', alpha=0.5)
axes[1].axvline(n_95, color='black', linestyle=':', alpha=0.5)
axes[1].set_xlabel('Number of Features')
axes[1].set_ylabel('Cumulative Importance')
axes[1].set_title('Cumulative Feature Importance', fontweight='bold')
axes[1].legend(fontsize=9)
axes[1].text(n_80 + 0.5, 0.75, f'{n_80} features', fontsize=8, color='grey')
axes[1].text(n_95 + 0.5, 0.90, f'{n_95} features', fontsize=8, color='black')

plt.tight_layout()
plt.show()

print(f'\n  {n_80} features explain 80% of model importance')
print(f'  {n_95} features explain 95% of model importance')
print(f'  Top feature: {imp_df.iloc[0]["feature"]}  ({imp_df.iloc[0]["importance"]:.4f})')

---
## 8. Agent Performance Profile

In [ ]:
# ── latency percentiles ────────────────────────────────────────────────────────
lat = scored_df['processing_time_ms']
percentiles = [50, 75, 90, 95, 99]
lat_stats = {f'p{p}': round(np.percentile(lat, p), 3) for p in percentiles}
lat_stats['mean'] = round(lat.mean(), 3)
lat_stats['max']  = round(lat.max(), 3)

print('Latency Profile (ms):')
for k, v in lat_stats.items():
    print(f'  {k:6s}: {v} ms')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# ── latency histogram ─────────────────────────────────────────────────────────
axes[0].hist(lat, bins=40, color='steelblue', edgecolor='white', alpha=0.85)
for p, color in [(50, 'green'), (95, 'orange'), (99, 'red')]:
    v = np.percentile(lat, p)
    axes[0].axvline(v, color=color, linestyle='--', linewidth=1.2,
                    label=f'p{p} = {v:.2f} ms')
axes[0].set_title('Scoring Latency Distribution', fontweight='bold')
axes[0].set_xlabel('ms / event')
axes[0].set_ylabel('Count')
axes[0].legend(fontsize=8)

# ── throughput: rolling 50-event avg prob ─────────────────────────────────────
scored_df_s = scored_df.sort_values('timestamp').reset_index(drop=True)
scored_df_s['roll_prob'] = scored_df_s['churn_probability'].rolling(50, min_periods=1).mean()
axes[1].plot(scored_df_s.index, scored_df_s['roll_prob'],
             color='#9b59b6', linewidth=2)
axes[1].axhline(THRESHOLD, color='red', linestyle='--',
                linewidth=1, label=f'Threshold = {THRESHOLD}')
axes[1].set_title('Rolling Avg Churn Probability (window=50)', fontweight='bold')
axes[1].set_xlabel('Event #')
axes[1].set_ylabel('P(Churn)')
axes[1].legend(fontsize=8)

plt.tight_layout()
plt.show()

---
## 9. Final Summary Report

In [ ]:
print()
print('╔' + '═'*60 + '╗')
print('║  REAL-TIME CHURN ANALYSIS AGENT — FINAL SUMMARY        ║')
print('╠' + '═'*60 + '╣')
print('║  DATASET                                                ║')
print(f'║    Telco Customer Churn  ·  {len(df_features):,} customers                ║')
print(f'║    Churn rate            ·  {df_features["Churn"].mean()*100:.1f}%                       ║')
print('╠' + '═'*60 + '╣')
print('║  MODEL                                                  ║')
print(f'║    Tuned Random Forest (GridSearchCV, F1-optimised)     ║')
print(f'║    Best params: {str(model_meta["best_params"]):40s}  ║')
print(f'║    ROC-AUC   : {model_meta["test_metrics"]["roc_auc"]}                              ║')
print(f'║    F1 score  : {model_meta["test_metrics"]["f1"]}                              ║')
print(f'║    Recall    : {model_meta["test_metrics"]["recall"]}   (critical for churn)        ║')
print(f'║    Precision : {model_meta["test_metrics"]["precision"]}                              ║')
print(f'║    Decision threshold  :  {THRESHOLD}                       ║')
print('╠' + '═'*60 + '╣')
print('║  REAL-TIME STREAM SIMULATION                            ║')
print(f'║    Events processed    :  {stream_summary["total_events"]:,}                         ║')
print(f'║    Alerts fired        :  {stream_summary["alerts_fired"]:,}  ({stream_summary["alert_rate_pct"]}%)           ║')
print(f'║    CRITICAL alerts     :  {stream_summary["critical_count"]}                           ║')
print(f'║    Avg scoring latency :  {stream_summary["avg_latency_ms"]} ms / event           ║')
print('╠' + '═'*60 + '╣')
print('║  BUSINESS IMPACT (estimated)                            ║')
print(f'║    Intervention cost   :  ${cost_of_intervention:,.0f}                    ║')
print(f'║    Revenue protected   :  ${revenue_saved:,.0f}                    ║')
print(f'║    Net benefit         :  ${net_benefit:,.0f}                    ║')
print(f'║    ROI                 :  {roi_pct:.1f}%                         ║')
print('╠' + '═'*60 + '╣')
print('║  TOP CHURN DRIVERS                                      ║')
for i, feat in enumerate(imp_df.head(5)['feature'], 1):
    print(f'║    {i}. {feat:50s}  ║')
print('╚' + '═'*60 + '╝')

In [ ]:
# ── save summary as JSON ──────────────────────────────────────────────────────
final_summary = {
    'dataset': {
        'total_customers': len(df_features),
        'churn_rate_pct' : round(df_features['Churn'].mean() * 100, 2)
    },
    'model': model_meta,
    'stream': stream_summary,
    'business_impact': {
        'intervention_cost_usd': cost_of_intervention,
        'revenue_protected_usd': revenue_saved,
        'net_benefit_usd'      : net_benefit,
        'roi_pct'              : round(roi_pct, 1)
    },
    'top_5_features': imp_df.head(5)['feature'].tolist()
}

with open('../data/final_summary.json', 'w') as f:
    json.dump(final_summary, f, indent=2)

print('✅  Final summary saved → data/final_summary.json')
print('\n🎉  All notebooks complete — project ready for presentation!')

---
## ✅ End-to-End Pipeline Recap

```
telco_churn.csv
      │
      ▼
01_data_inspection_cleaning.ipynb
  └─► telco_churn_clean.csv
      │
      ▼
02_feature_engineering.ipynb
  └─► telco_churn_features.csv  ·  scaler.pkl  ·  feature_metadata.json
      │
      ▼
03_churn_prediction_model.ipynb
  └─► best_model.pkl  ·  model_metadata.json
      │
      ▼
04_realtime_alert_simulation.ipynb
  └─► events_stream.jsonl  ·  alert_log.csv  ·  scored_all.csv  ·  stream_summary.json
      │
      ▼
05_result_analysis.ipynb
  └─► final_summary.json  (this notebook)
```